# Porting the Regressor from Wang et al.

The paper uses an XGBoost regressor trained on the gene expression of expert annotated slides.Gene features are selected, then transformed with PCA. These rotated expressions are fed into XGBoost regressor. They performed Leave-on-Out testing to find the best hyperparameters, then trained on the entire annotated dataset.

The trained regressor is then used to predict annotations for the un-annotated slides. 

In [1]:
import polars as pl
import pandas as pd
import anndata as ad
import numpy as np
import scanpy as sc

from anndata import AnnData
from anndata.experimental import AnnCollection

from pathlib import Path

## Training the Regressor

## Preparing Data
First, read in and prepare the expression data and annotations as in their R code. 

In [2]:
# Only certain slides are annotated to the proper level
# There are listed in this file
df_ids = pl.read_csv('../data/clinical-ids.csv')

Combine physician annotations into three main categories: Tumor, Stroma, and Lympocyte. These provide the basic categories for their later analysis (tumor vs non-tumor)

In [3]:
def combine_annotations(ann: AnnData) -> AnnData:
    '''
    Combine the clinical annotations to the categories used
    in the study's R code
    '''

    combined = {
        'Tumor': [
            (1, 'Tumor'), 
            (1, 'Tumor region')
        ],
        'Stroma': [
            (1, 'Stroma cell'), 
            (1, 'Low TIL stroma'),
            (0.5, 'High TIL stroma'), 
            (1, 'Acellular stroma')
        ],
        'Lymphocyte': [
            (1, 'Lymphocyte'),
            (0.5, 'High TIL stroma')
        ]
    }

    # aggregate some categories
    df = pd.DataFrame()
    df.index = ann.obs.index

    for category, components in combined.items():
        df[category] = np.zeros(len(ann.obs))
        for factor, col in components:
            df[category] += factor * ann[:, col].X.flatten()

    # drop no cell and the components of the combined categories
    drops = ["Nothing", "Artefacts", "Hole (whitespace)"] + \
        [col for comps in combined.values() for _, col in comps]
    df = pd.concat([df, ann.to_df().drop(columns=drops)], axis=1)

    # normalize 
    df_frac = df.div(df.sum(axis=1), axis=0)
    df_frac.fillna(0, inplace=True)

    # add as observation matrix to anndata
    # the raw counts are there for later filtering
    ann.obsm['combined_raw'] = df
    ann.obsm['combined_frac'] = df_frac
    
    return ann

Then, read in all needed counts and annotation files. Pull out only those spots on slides that were annotated, and add the annotation info to the counts.

In [4]:
def add_annotation_to_counts(cnt: AnnData, anno: AnnData) -> AnnData:
    '''
    For easy of use, add an order matrix of the annotation score directly 
    to the count AnnData
    '''
    idx = ['x', 'y']
    
    def join_and_reorder(df: pd.DataFrame):
        # ensure that the annotation scores match row-to-row (spot) for the counts
        df = pd.merge(
            cnt.obs[idx],
            df,
            on=idx,
            how='left'
        )
        # add count slide name indices
        df.index = cnt.obs.index
        # drop ordering indices and add to count AnnData
        return df.drop(columns=idx)
    
    matrices = {
        'annotation': anno.obsm['combined_frac'].join(anno.obs[idx]),
        'raw_annotation': anno.obsm['combined_raw'].join(anno.obs[idx])
    }

    for key, df in matrices.items():
        cnt.obsm[key] = join_and_reorder(df)

    return cnt

def read_annotated_counts(ids: pl.DataFrame, base_anno: Path, base_cnt: Path):
    def filter_cnt_ad(a: AnnData, slide: str):
        return a[a.obs['slide'] == slide].copy()

    f_anno = (pl.lit(f'{base_anno}/annotation-') + 
        pl.col('id').cast(pl.String) + 
        pl.lit('.h5ad')).alias('f_anno')
    
    f_cnt = (pl.lit(f'{base_cnt}/cnt-') + 
        pl.col('id').cast(pl.String) + 
        pl.lit('.h5ad')).alias('f_cnt')
    
    df = (ids
        .filter('hasAnnot')
        .select('id', 'names')
        .with_columns(f_anno, f_cnt)
    )

    cnts = [
        add_annotation_to_counts(
            filter_cnt_ad(ad.read_h5ad(fcnt), slide),
            combine_annotations(ad.read_h5ad(fanno))
        )
        for (_id, slide, fanno, fcnt) in df.iter_rows()
    ]

    keys = df.select('id').to_series()

    return ad.concat(cnts, axis='obs', join='inner', label='tnbc_id', keys=keys)

In [5]:
BASE_ANNO = Path('../data/anndata/annotations')
BASE_CNTS = Path('../data/anndata/counts')

annocnt = read_annotated_counts(df_ids, BASE_ANNO, BASE_CNTS)

In [6]:
display(annocnt)

AnnData object with n_obs × n_vars = 96706 × 14019
    obs: 'x', 'y', 'new_x', 'new_y', 'pixel_x', 'pixel_y', 'slide', 'tnbc_id'
    obsm: 'annotation', 'raw_annotation'

### Normalization and Filtering Genes and Spots

The paper normalized the genes with CP10K and lop1p. It then filtered to 4,000 genes based on some basic counts, and if that wasn't enough, variance. I performed the count normalization, but then used `scanpy`'s default gene selection to pick 

Spots will low total (< 1000) annotated pixels were removed as well.

In [7]:
def normalize_counts(a: AnnData):
    '''
    Use scanpy to perfrom CP10K normalization and lop1p scaling
    like in the provided R code.

    '''
    sc.pp.normalize_total(a, target_sum=10_000, inplace=True)
    sc.pp.log1p(a)

def select_genes(a: AnnData, num_genes=1000):
    '''
    I can't find the exact method that was used, but 
    hopefully Seurat [Satija et al., 2015] is close enough.

    Note that this has to run on the entire dataset (the `AnnCollection`),
    not on an individual sample.
    '''

    sc.pp.highly_variable_genes(a, n_top_genes=num_genes)

def filter_spots(ann: AnnData, total_pixels=1000):
    ''' 
    Filter to counts to have only spots with greater than
    `total_pixels` of annotation.
    Store as a column in `obs`
    '''
    ann.obs['highly_annotated'] = ann.obsm['raw_annotation'].sum(axis=1) > total_pixels
    return ann

def norm_and_filter(ann: AnnData) -> AnnData:
    '''
    This roughly follows what their code performed,
    including the choice of 4,000 variable genes for PCA reduction
    '''
    ann = ann.copy()

    normalize_counts(ann)
    select_genes(ann, 4000)
    return filter_spots(ann)    
    

In [8]:
acnt_filtered = norm_and_filter(annocnt)

display(acnt_filtered)
display(acnt_filtered.X)

AnnData object with n_obs × n_vars = 96706 × 14019
    obs: 'x', 'y', 'new_x', 'new_y', 'pixel_x', 'pixel_y', 'slide', 'tnbc_id', 'highly_annotated'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'log1p', 'hvg'
    obsm: 'annotation', 'raw_annotation'

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 273642624 stored elements and shape (96706, 14019)>

## PCA and Training XGBoost
They calculate 500 PCA dimensions, then only use 250 for the regressor. 

As the counts are in a sparse matrix, use `sklearn`'s `TruncatedSVD` to perfrom dimensionality reduction.

Once the "PCA"s are calculated, filter out low annotation spots. Then, train the regressor.

In [9]:
from sklearn.decomposition import TruncatedSVD
import xgboost as xgb

In [10]:
def train_svd(ann: AnnData, n=250, var_filter='highly_variable'):
    svd = TruncatedSVD(n_components=n, random_state=42)
    # interestingly, when they perform the PCA, 
    # they use all spots, not just the spots with lots of pixels
    svd.fit(ann[:, ann.var[var_filter]].X)

    return svd

def transform_with_svd(svd: TruncatedSVD, ann: AnnData, key = 'X_pca', var_filter='highly_variable'):
    X = ann[:, ann.var[var_filter]].X
    ann.obsm[key] = svd.transform(X)

    return ann


In [11]:
def train_xgb_regressor(ann: AnnData, category: str):
    param = {
        'booster': 'gblinear',
        'objective': 'reg:logistic',
    }
    rounds = 25

    dtrain = xgb.DMatrix(
        ann.obsm['X_pca'],
        ann.obsm['annotation'][category],
    )
    # save training matrices?

    return xgb.train(param, dtrain, num_boost_round=rounds)

def train_regressors(ann: AnnData) -> dict[str, xgb.Booster]:
    '''
    Train one XGBRegressor for each annotation category
    '''
    categories = ann.obsm['annotation'].columns.to_list()
    # filter to only high pixel annotations
    ann = ann[ann.obs['highly_annotated'], :]

    return {
        category: train_xgb_regressor(ann, category)
        for category in categories
    }


class AnnoRegressor():
    '''
    Collect the various parts need to run the regression
    on all spots
    This is trained on the annotated slide spots
    '''
    data: AnnData
    features: str
    svd: TruncatedSVD
    regressors: dict[str, xgb.Booster]

    def __init__(self, ann: AnnData):
        '''
        Reduce the gene counts dimensionality with SVD.
        Then, train one xgb booster for each annotation category.
        '''
        svd = train_svd(ann, n = 250)
        print('trained SVD')
        ann = transform_with_svd(svd, ann)
        features = ann.var_names[ann.var['highly_variable']].to_list()
        regressors = train_regressors(ann)
        print(f'trained {len(regressors)} regressors')

        self.data = ann
        self.features = features
        self.svd = svd
        self.regressors = regressors

    def train_svd(self):
        pass

    def regress_sample(self, sample, polars = True) -> pl.DataFrame | pd.DataFrame:
        # select only genes used
        filtered = sample[:, self.features]

        # return filtered
        projected = self.svd.transform(filtered.X)
        dpredict = xgb.DMatrix(projected)
        res = {
            category: reg.predict(dpredict)
            for category, reg in self.regressors.items()
        }

        # add observation / spot info
        # TODO: make this work with pd or obs
        obs = filtered.obs.df
        df = pd.concat([pd.DataFrame(res, index=obs.index), obs], axis=1).reset_index(names='slide_id')

        return pl.from_dataframe(df) if polars else df

In [12]:
with xgb.config_context(verbosity=2):
    regressor = AnnoRegressor(acnt_filtered)

trained SVD
trained 11 regressors


## Predicting Annotation Based on Gene Expression
Now, all of the spots can have their annotations predicted with the trained `AnnoRegressor` class. Read in the counts for each sample, predict the annotations, and save out a dataframe.

In [13]:
def read_counts(dir: Path) -> AnnCollection:
    '''
    Read in all of the sample count files,
    and normalize (CP10K + log1p)
    '''
    def read_file_get_id(fname: Path):
        id = int(fname.stem.split('-')[-1])
        adata = ad.read_h5ad(fname)
        normalize_counts(adata)
        return (id, adata)

    files = dict(map(read_file_get_id, dir.rglob("*.h5ad")))

    return AnnCollection(files, join_vars='inner', label='tnbc_id')

def regress_counts(regressor: AnnoRegressor, counts: AnnCollection) -> pl.DataFrame:
    '''
    Using the normalized counts for each spot, get the logistic regression for each
    of the annotation categories
    '''
    output = None

    for batch, idx in counts.iterate_axis(50_000):
        display(f'regressing spots {idx[0]} to {idx[-1]}')
        temp = regressor.regress_sample(batch)

        if output is None:
            output = temp
        else:
            output = pl.concat([output, temp], how='vertical_relaxed')

    # move some columns around in output
    order = [
        'tnbc_id',
        'slide_id',
        'slide',
        'x',
        'y',
        'Tumor',
        'Stroma',
        'Lymphocyte',
        'Necrosis',
        'Fat tissue',
        'Vessels',
        'Lactiferous duct',
        'in situ',
        'Lymphoid nodule',
        'Nerve',
        'Heterologous elements',
        'new_x',
        'new_y',
        'pixel_x',
        'pixel_y',
    ]

    return output.select(order).sort(order[:1] + order[2:5])


In [14]:
all_counts = read_counts(BASE_CNTS)
display(all_counts)

AnnCollection object with n_obs × n_vars = 270310 × 14019
  constructed from 94 AnnData objects
    obs: 'x', 'y', 'new_x', 'new_y', 'pixel_x', 'pixel_y', 'slide', 'tnbc_id'

In [15]:
df_class = regress_counts(regressor, all_counts)

'regressing spots 0 to 49999'

'regressing spots 50000 to 99999'

'regressing spots 100000 to 149999'

'regressing spots 150000 to 199999'

'regressing spots 200000 to 249999'

'regressing spots 250000 to 270309'

In [16]:
df_class

tnbc_id,slide_id,slide,x,y,Tumor,Stroma,Lymphocyte,Necrosis,Fat tissue,Vessels,Lactiferous duct,in situ,Lymphoid nodule,Nerve,Heterologous elements,new_x,new_y,pixel_x,pixel_y
i64,str,str,i32,i32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f64,f64,f64,f64
1,"""CN1_C1.2x12""","""CN1_C1""",2,12,0.051512,0.710642,0.011412,0.005507,0.535842,0.000157,0.000109,9.6848e-8,1.8558e-8,2.8373e-8,1.8024e-10,2.04,12.05,197.507095,179.361456
1,"""CN1_C1.2x14""","""CN1_C1""",2,14,0.009688,0.808986,0.007917,0.003309,0.258634,0.001083,0.016385,0.000057,1.5544e-7,1.1321e-7,1.0611e-12,2.05,14.08,197.594595,209.898956
1,"""CN1_C1.2x16""","""CN1_C1""",2,16,0.010854,0.74208,0.006845,0.002218,0.450321,0.004585,0.004819,0.000022,7.5523e-8,1.7265e-11,1.6628e-14,2.03,16.05,197.294595,239.486456
1,"""CN1_C1.2x18""","""CN1_C1""",2,18,0.022696,0.528092,0.01062,0.019778,0.42676,0.001712,0.002673,0.000004,4.2941e-10,4.9204e-8,3.2793e-11,2.03,18.04,197.307095,269.548956
1,"""CN1_C1.2x20""","""CN1_C1""",2,20,0.032434,0.790398,0.02224,0.006685,0.151887,0.001451,0.001902,0.000128,3.6626e-8,2.5435e-10,1.3721e-12,2.07,20.04,197.857095,299.623956
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
96,"""CN48_E2.65x37""","""CN48_E2""",65,37,0.711262,0.337234,0.013664,0.006525,0.004319,0.005801,0.002256,0.001384,0.000004,1.5856e-8,1.7459e-11,65.02,36.99,137.111256,405.595386
96,"""CN48_E2.65x39""","""CN48_E2""",65,39,0.761428,0.275821,0.013307,0.012923,0.00188,0.002901,0.002184,0.002312,0.000005,1.9217e-8,6.2076e-10,65.01,38.97,133.965915,376.111925
96,"""CN48_E2.65x41""","""CN48_E2""",65,41,0.677267,0.29499,0.013234,0.038146,0.005223,0.002465,0.00085,0.00073,0.000008,3.8654e-9,2.5976e-10,65.03,40.98,130.283535,346.110931


In [17]:
df_class.write_csv('../data/predicted_classifications.csv')

## Comparison to Paper Annotations
Align classification results from paper with our predictions.
Calculate an RMSE for each category, as well as the overall RMSE.

Maybe I should use the RMSE from sklearn for each regressor?

In [ ]:
df_paper = pl.read_csv('../data/all_classifications.csv')

In [19]:
df_paper

tnbc_id,slide_id,pixel_x,pixel_y,slide_rep,x,y,Fat tissue,Heterologous elements,in situ,Lactiferous duct,Lymphocyte,Lymphoid nodule,Necrosis,Nerve,Stroma,Tumor,Vessels
i64,str,f64,f64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1,"""CN1_C1.2x12""",39.501419,35.872291,1,2,12,0.578129,0.000007,0.000004,0.001398,0.020524,0.00003,0.020807,0.000635,0.613463,0.045485,0.001069
1,"""CN1_C1.2x14""",39.518919,41.979791,1,2,14,0.510407,0.000007,0.000038,0.01032,0.015268,0.000131,0.029339,0.000032,0.864973,0.003384,0.02552
1,"""CN1_C1.2x16""",39.458919,47.897291,1,2,16,0.715456,0.00001,0.000046,0.000758,0.015701,0.000429,0.009174,8.4059e-7,0.675071,0.011369,0.001028
1,"""CN1_C1.2x18""",39.461419,53.909791,1,2,18,0.75413,0.000016,0.000022,0.001173,0.010491,0.000026,0.016051,0.000626,0.664755,0.014724,0.001533
1,"""CN1_C1.2x20""",39.571419,59.924791,1,2,20,0.262702,0.000005,0.000098,0.005349,0.038164,0.000037,0.012984,0.000045,0.872827,0.02036,0.007695
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
96,"""CN48_E2.65x37""",27.422251,81.119077,3,65,37,0.000643,0.000108,0.002203,0.003731,0.0076,0.000247,0.004335,0.000019,0.298671,0.81906,0.001473
96,"""CN48_E2.65x39""",26.793183,75.222385,3,65,39,0.000753,0.000022,0.001122,0.002997,0.006978,0.000119,0.004767,0.000001,0.190243,0.829741,0.000987
96,"""CN48_E2.65x41""",26.056707,69.222186,3,65,41,0.002384,0.000061,0.001784,0.007077,0.008733,0.000227,0.037683,0.000001,0.199893,0.725897,0.00091


In [20]:
df_compare = df_paper.drop('pixel_x', 'pixel_y', 'slide_rep') \
    .join(
        df_class.drop('pixel_x', 'pixel_y', 'slide', 'new_x', 'new_y'),
        on=['tnbc_id', 'slide_id', 'x', 'y'],
        how='left',
        suffix='_predict',
    )

df_compare.shape

(270310, 26)

In [21]:
def calc_category_rmse(df: pl.DataFrame):
    predict_cols = [c for c in df.columns if c.endswith("_predict")]

    pairs = [
        (col.replace("_predict", ""), col)
        for col in predict_cols
    ]

    rmses = [
        ((pl.col(label) - pl.col(predict)) ** 2)
        .mean()
        .sqrt()
        .alias(label)

        for label, predict in pairs
    ]


    return df.select(rmses).unpivot(variable_name='Category', value_name='RMSE')

In [22]:
def calc_overall_rmse(df: pl.DataFrame):
    predict_cols = [c for c in df.columns if c.endswith("_predict")]
    pairs = [
        (col.replace("_predict", ""), col)
        for col in predict_cols
    ]

    labels = pl.concat_list(pl.col(label) for label, _ in pairs).alias('label')
    predicts = pl.concat_list(pl.col(pred) for _, pred in pairs).alias('predict')

    df = df.lazy().select(labels, predicts).explode('label', 'predict').collect()
    

    return float(np.sqrt(((df['label'] - df['predict']) ** 2).mean()))

In [23]:
df_cat_rmse = calc_category_rmse(df_compare)
overall_rmse = calc_overall_rmse(df_compare)

In [24]:
with pl.Config(tbl_rows=11, float_precision=3):
    display(df_cat_rmse)

display(f'Overall RMSE:  {overall_rmse:0.3f}')

Category,RMSE
str,f64
"""Tumor""",0.079
"""Stroma""",0.099
"""Lymphocyte""",0.040
"""Necrosis""",0.063
"""Fat tissue""",0.066
"""Vessels""",0.014
"""Lactiferous duct""",0.015
"""in situ""",0.016
"""Lymphoid nodule""",0.022


'Overall RMSE:  0.050'

Pretty good RMSE as compared to the paper's predictions. As RMSE is "in the same units" as the input–which is logistic from 0 to 1–the errors range from ~1% to 10%.